# **Installation**

In [1]:
!pip install -q lightgbm catboost imbalanced-learn xgboost scikit-learn pandas numpy
!pip install ipython-autotime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 71.7 MB/s eta 0:00:00


# **Import Libraries**

In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from scipy.stats import uniform

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (make_scorer, accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, matthews_corrcoef, cohen_kappa_score, confusion_matrix, balanced_accuracy_score)
from imblearn.metrics import geometric_mean_score
from sklearn.model_selection import KFold
from imblearn.over_sampling import ADASYN
import time
from scipy.stats import uniform
from sklearn.decomposition import PCA

# **Load Dataset**

In [3]:
df = pd.read_csv('/content/INCART 2-lead Arrhythmia Database.csv')

# **Data Preprocessing**

In [4]:
# Drop Leakage Feature
if 'record' in df.columns:
    df.drop('record', axis=1, inplace=True)

# Handle Null Values
for col in df.columns:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

df=df[df['type'] != 'Q']

# 4. Encode Target
df['type'] = df['type'].map({'F': 0, 'N': 1, 'SVEB': 2, 'VEB': 3})

# Split Features & Target
X = df.drop('type', axis=1)
y = df['type']
feature_names = X.columns

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# **Scaling**


In [5]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)


# **Sampling Technique**

In [6]:
adasyn = ADASYN(random_state=42)
X_train_ad, y_train_ad = adasyn.fit_resample(X_train_s, y_train)

In [7]:
pca = PCA(n_components=0.95, random_state=42)
X_train_pca = pca.fit_transform(X_train_ad)
X_test_pca = pca.transform(X_test_s)

# **Cross Validation Setup**

In [8]:
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# **Evaluation Function**

In [9]:
results = []

def run_model_evaluation(model_name, clf_model):
    global results
    fold_wise_data = []

    for i, (train_idx, test_idx) in enumerate(skf.split(X_train_pca, y_train_ad)):

        X_tr, X_tes = X_train_pca[train_idx], X_train_pca[test_idx]
        y_tr, y_tes = y_train_ad[train_idx], y_train_ad[test_idx]

        # Training start
        t_start = time.time()
        clf_model.fit(X_tr, y_tr)
        t_end = time.time()

        # Predictions
        preds = clf_model.predict(X_tes)
        probs = clf_model.predict_proba(X_tes)

        cm = confusion_matrix(y_tes, preds)
        fold_recalls = []
        fold_specificities = []

        for c in range(4):
            tp = cm[c, c]
            fn = np.sum(cm[c, :]) - tp
            fp = np.sum(cm[:, c]) - tp
            tn = np.sum(cm) - (tp + fp + fn)

            fold_recalls.append(tp / (tp + fn) if (tp + fn) > 0 else 0)
            fold_specificities.append(tn / (tn + fp) if (tn + fp) > 0 else 0)

        re = np.mean(fold_recalls)
        spec = np.mean(fold_specificities)

        # Dictionary for each fold
        metrics = {
            "Fold": i + 1,
            "Classifier": model_name,
            "Accuracy": round(accuracy_score(y_tes, preds), 4),
            "Precision": round(precision_score(y_tes, preds, average='weighted', zero_division=0), 4),
            "Recall": round(re, 4),
            "Specificity": round(spec, 4),
            "G-Mean": round(np.sqrt(re * spec), 4),
            "F1-Score": round(f1_score(y_tes, preds, average='weighted'), 4),
            "MCC": round(matthews_corrcoef(y_tes, preds), 4),
            "Kappa": round(cohen_kappa_score(y_tes, preds), 4),
            "ROC-AUC": round(roc_auc_score(y_tes, probs, multi_class='ovr', average='weighted'), 4),
            "Balanced Accuracy": round(balanced_accuracy_score(y_tes, preds), 4),
            "Train_Time": round(t_end - t_start, 4)
        }
        fold_wise_data.append(metrics)

    # Result
    df_fold = pd.DataFrame(fold_wise_data)
    print(f"--- For: {model_name} ---")
    display(df_fold)

    results.extend(fold_wise_data)

# **LogisticRegression**

In [31]:
from sklearn.linear_model import LogisticRegression
#  Model base initialize
model = LogisticRegression(max_iter=2000, random_state=42)

# Parameter distribution setup
params = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'solver': ['liblinear', 'lbfgs'],
    'penalty': ['l2']
}

#  Randomized Search execution
rs = RandomizedSearchCV(
    estimator=model,
    param_distributions=params,
    n_iter=10,
    cv=skf,
    scoring='f1_weighted',
    n_jobs=-1,
    random_state=42
)

# Training
rs.fit(X_train_pca, y_train_ad)

# Best model extract
final_lr_model = rs.best_estimator_

print(f"Best Parameters: {rs.best_params_}")

run_model_evaluation("Logistic Regression", final_lr_model)

Best Parameters: {'solver': 'lbfgs', 'penalty': 'l2', 'C': 10}
--- For: Logistic Regression ---


,Fold,Classifier,Accuracy,Precision,Recall,Specificity,G-Mean,F1-Score,MCC,Kappa,ROC-AUC,Balanced Accuracy,Train_Time
0,1,Logistic Regression,0.6770,0.6784,0.6770,0.8923,0.7772,0.6765,0.5700,0.5693,0.8938,0.6770,6.5502
1,2,Logistic Regression,0.6791,0.6804,0.6791,0.8930,0.7787,0.6787,0.5727,0.5721,0.8937,0.6791,5.4004
2,3,Logistic Regression,0.6764,0.6778,0.6764,0.8921,0.7768,0.6762,0.5691,0.5685,0.8922,0.6764,6.0880
3,4,Logistic Regression,0.6782,0.6793,0.6782,0.8927,0.7781,0.6779,0.5713,0.5709,0.8932,0.6782,5.2720
4,5,Logistic Regression,0.6784,0.6799,0.6784,0.8928,0.7782,0.6779,0.5719,0.5712,0.8940,0.6784,4.9309
5,6,Logistic Regression,0.6769,0.6785,0.6769,0.8923,0.7772,0.6765,0.5699,0.5692,0.8924,0.6769,6.3088
6,7,Logistic Regression,0.6738,0.6754,0.6738,0.8913,0.7750,0.6735,0.5658,0.5651,0.8914,0.6738,5.0186
7,8,Logistic Regression,0.6772,0.6787,0.6772,0.8924,0.7774,0.6768,0.5702,0.5696,0.8923,0.6772,6.8385
8,9,Logistic Regression,0.6771,0.6784,0.6771,0.8924,0.7773,0.6766,0.5701,0.5694,0.8919,0.6771,5.1197
9,10,Logistic Regression,0.6742,0.6755,0.6742,0.8914,0.7752,0.6738,0.5662,0.5656,0.8912,0.6742,6.2808


# **Decision Tree**

In [30]:
from sklearn.tree import DecisionTreeClassifier
#  Model base initialize
model = DecisionTreeClassifier(random_state=42)

# Parameter distribution setup
params = {
    "criterion": ['gini', 'entropy'],
    "max_depth": [None, 5, 10, 15, 20],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 5, 10],
    "max_features": ['None', 'sqrt', 'log2']

}

#  Randomized Search execution
rs = RandomizedSearchCV(
    estimator=model,
    param_distributions=params,
    n_iter=10,
    cv=skf,
    scoring='f1_weighted',
    n_jobs=-1,
    random_state=42
)

# Training
rs.fit(X_train_pca, y_train_ad)

# Best model extract
final_dt_model = rs.best_estimator_

print(f"Best Parameters: {rs.best_params_}")

run_model_evaluation("Decision Tree", final_dt_model)

Best Parameters: {'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 'log2', 'max_depth': None, 'criterion': 'entropy'}
--- For: Decision Tree ---


,Fold,Classifier,Accuracy,Precision,Recall,Specificity,G-Mean,F1-Score,MCC,Kappa,ROC-AUC,Balanced Accuracy,Train_Time
0,1,Decision Tree,0.9932,0.9932,0.9932,0.9977,0.9955,0.9932,0.9910,0.9910,0.9969,0.9932,8.0719
1,2,Decision Tree,0.9930,0.9930,0.9930,0.9977,0.9953,0.9930,0.9906,0.9906,0.9966,0.9930,8.5748
2,3,Decision Tree,0.9932,0.9932,0.9932,0.9977,0.9955,0.9932,0.9909,0.9909,0.9967,0.9932,7.5546
3,4,Decision Tree,0.9931,0.9931,0.9931,0.9977,0.9954,0.9931,0.9908,0.9908,0.9970,0.9931,8.0883
4,5,Decision Tree,0.9938,0.9938,0.9938,0.9979,0.9959,0.9938,0.9917,0.9917,0.9971,0.9938,7.2933
5,6,Decision Tree,0.9928,0.9928,0.9928,0.9976,0.9952,0.9928,0.9904,0.9904,0.9965,0.9928,7.6829
6,7,Decision Tree,0.9918,0.9918,0.9918,0.9973,0.9945,0.9918,0.9891,0.9891,0.9961,0.9918,6.9835
7,8,Decision Tree,0.9927,0.9926,0.9927,0.9976,0.9951,0.9926,0.9902,0.9902,0.9965,0.9927,8.4890
8,9,Decision Tree,0.9929,0.9929,0.9929,0.9976,0.9952,0.9929,0.9905,0.9905,0.9967,0.9929,7.9300
9,10,Decision Tree,0.9917,0.9917,0.9917,0.9972,0.9945,0.9917,0.9890,0.9890,0.9961,0.9917,7.4139


# **Random Forest**

In [26]:
from sklearn.ensemble import RandomForestClassifier
#  Model base initialize
model = RandomForestClassifier(random_state=42, n_jobs=-1)

# Parameter distribution setup
params = {
  "n_estimators": [50, 100],
  "max_depth": [10, 15],
  "min_samples_split": [5],
  "max_features": ['sqrt']
}

#  Randomized Search execution
rs = RandomizedSearchCV(
    estimator=model,
    param_distributions=params,
    n_iter=2,
    cv=3,
    scoring='f1_weighted',
    n_jobs=-1,
    random_state=42
)

# Training
rs.fit(X_train_pca, y_train_ad)

# Best model extract
final_rf_model = rs.best_estimator_

print(f"Best Parameters: {rs.best_params_}")

run_model_evaluation("Random Forest", final_rf_model)

Best Parameters: {'n_estimators': 100, 'min_samples_split': 5, 'max_features': 'sqrt', 'max_depth': 15}
--- For: Random Forest ---


,Fold,Classifier,Accuracy,Precision,Recall,Specificity,G-Mean,F1-Score,MCC,Kappa,ROC-AUC,Balanced Accuracy,Train_Time
0,1,Random Forest,0.9966,0.9966,0.9966,0.9989,0.9977,0.9966,0.9954,0.9954,0.9999,0.9966,229.4989
1,2,Random Forest,0.9968,0.9969,0.9968,0.9989,0.9979,0.9968,0.9958,0.9958,0.9999,0.9968,229.7285
2,3,Random Forest,0.9969,0.9969,0.9969,0.9990,0.9979,0.9969,0.9959,0.9959,0.9999,0.9969,236.4537
3,4,Random Forest,0.9962,0.9962,0.9962,0.9987,0.9974,0.9962,0.9949,0.9949,1.0000,0.9962,238.0894
4,5,Random Forest,0.9965,0.9965,0.9965,0.9988,0.9976,0.9965,0.9953,0.9953,0.9999,0.9965,240.6508
5,6,Random Forest,0.9962,0.9962,0.9962,0.9987,0.9975,0.9962,0.9949,0.9949,0.9999,0.9962,236.8259
6,7,Random Forest,0.9966,0.9966,0.9966,0.9989,0.9977,0.9966,0.9954,0.9954,0.9999,0.9966,236.0334
7,8,Random Forest,0.9957,0.9957,0.9957,0.9986,0.9971,0.9957,0.9942,0.9942,0.9999,0.9957,234.2730
8,9,Random Forest,0.9960,0.9960,0.9960,0.9987,0.9973,0.9960,0.9947,0.9947,0.9999,0.9960,253.8382
9,10,Random Forest,0.9958,0.9958,0.9958,0.9986,0.9972,0.9958,0.9944,0.9944,0.9999,0.9958,260.9948


# **GBM**

In [25]:
from sklearn.ensemble import HistGradientBoostingClassifier
#  Model base initialize
model = HistGradientBoostingClassifier(random_state=42)

# Parameter distribution setup
params = {
  "max_iter": [50, 100],
  "learning_rate": [0.1, 0.2]
}

#  Randomized Search execution
rs = RandomizedSearchCV(
    estimator=model,
    param_distributions=params,
    n_iter=2,
    cv=3,
    scoring='f1_weighted',
    n_jobs=-1,
    random_state=42
)

# Training
rs.fit(X_train_pca, y_train_ad)

# Best model extract
final_gbm_model = rs.best_estimator_

print(f"Best Parameters: {rs.best_params_}")

run_model_evaluation("GBM", final_gbm_model)

Best Parameters: {'max_iter': 100, 'learning_rate': 0.1}
--- For: GBM ---


,Fold,Classifier,Accuracy,Precision,Recall,Specificity,G-Mean,F1-Score,MCC,Kappa,ROC-AUC,Balanced Accuracy,Train_Time
0,1,GBM,0.9969,0.9969,0.9969,0.9990,0.9979,0.9969,0.9959,0.9959,0.9998,0.9969,29.6705
1,2,GBM,0.9962,0.9962,0.9962,0.9987,0.9975,0.9962,0.9950,0.9950,0.9998,0.9962,29.5480
2,3,GBM,0.9966,0.9966,0.9966,0.9989,0.9977,0.9966,0.9955,0.9955,0.9998,0.9966,30.3526
3,4,GBM,0.9964,0.9964,0.9964,0.9988,0.9976,0.9964,0.9952,0.9952,0.9999,0.9964,29.5952
4,5,GBM,0.9968,0.9968,0.9968,0.9989,0.9979,0.9968,0.9957,0.9957,0.9998,0.9968,29.7362
5,6,GBM,0.9967,0.9967,0.9967,0.9989,0.9978,0.9967,0.9956,0.9955,0.9998,0.9967,29.6610
6,7,GBM,0.9964,0.9964,0.9964,0.9988,0.9976,0.9964,0.9952,0.9952,0.9997,0.9964,29.5092
7,8,GBM,0.9966,0.9966,0.9966,0.9989,0.9977,0.9966,0.9955,0.9955,0.9998,0.9966,30.5181
8,9,GBM,0.9962,0.9962,0.9962,0.9987,0.9974,0.9961,0.9949,0.9949,0.9998,0.9962,28.9919
9,10,GBM,0.9955,0.9956,0.9955,0.9985,0.9970,0.9955,0.9941,0.9941,0.9995,0.9955,28.4060


# **XGboost**

In [24]:
from xgboost import XGBClassifier
#  Model base initialize
model = XGBClassifier(eval_metric='logloss', use_label_encoder=False, random_state=42, tree_method='hist')

# Parameter distribution setup
params = {
"n_estimators": [50, 100],
"learning_rate": [0.1, 0.2],
"max_depth": [3, 5, 6],
"subsample": [0.8],
"colsample_bytree":[0.8]
}

#  Randomized Search execution
rs = RandomizedSearchCV(
    estimator=model,
    param_distributions=params,
    n_iter=2,
    cv=3,
    scoring='f1_weighted',
    n_jobs=-1,
    random_state=42
)

# Training
rs.fit(X_train_pca, y_train_ad)

# Best model extract
final_xg_model = rs.best_estimator_

print(f"Best Parameters: {rs.best_params_}")

run_model_evaluation("xgboost", final_xg_model)

Best Parameters: {'subsample': 0.8, 'n_estimators': 100, 'max_depth': 5, 'learning_rate': 0.2, 'colsample_bytree': 0.8}
--- For: xgboost ---


,Fold,Classifier,Accuracy,Precision,Recall,Specificity,G-Mean,F1-Score,MCC,Kappa,ROC-AUC,Balanced Accuracy,Train_Time
0,1,xgboost,0.9892,0.9892,0.9892,0.9964,0.9928,0.9891,0.9856,0.9855,0.9994,0.9892,19.8826
1,2,xgboost,0.9892,0.9893,0.9892,0.9964,0.9928,0.9892,0.9857,0.9856,0.9995,0.9892,22.0532
2,3,xgboost,0.9903,0.9903,0.9903,0.9968,0.9935,0.9903,0.9871,0.9871,0.9995,0.9903,22.2078
3,4,xgboost,0.9898,0.9898,0.9898,0.9966,0.9932,0.9898,0.9864,0.9864,0.9996,0.9898,19.8497
4,5,xgboost,0.9897,0.9897,0.9897,0.9966,0.9931,0.9896,0.9862,0.9862,0.9995,0.9897,21.7462
5,6,xgboost,0.9896,0.9897,0.9896,0.9965,0.9931,0.9896,0.9862,0.9862,0.9994,0.9896,22.5127
6,7,xgboost,0.9893,0.9893,0.9893,0.9964,0.9929,0.9893,0.9858,0.9857,0.9994,0.9893,19.7834
7,8,xgboost,0.9894,0.9894,0.9894,0.9965,0.9929,0.9894,0.9859,0.9858,0.9996,0.9894,21.9318
8,9,xgboost,0.9898,0.9898,0.9898,0.9966,0.9932,0.9898,0.9864,0.9864,0.9995,0.9898,22.1641
9,10,xgboost,0.9893,0.9894,0.9893,0.9964,0.9929,0.9893,0.9858,0.9858,0.9995,0.9893,18.6386


# **LightGBM**

In [10]:
from lightgbm import LGBMClassifier
#  Model base initialize
model = LGBMClassifier(random_state=42,verbosity=-1)

# Parameter distribution setup
params = {
"n_estimators": [100, 150],
"learning_rate": [0.05, 0.1],
"num_leaves": [31, 40],
"boosting_type": ['gbdt']
}

#  Randomized Search execution
rs = RandomizedSearchCV(
    estimator=model,
    param_distributions=params,
    n_iter=2,
    cv=3,
    scoring='f1_weighted',
    n_jobs=-1,
    random_state=42
)

# Training
rs.fit(X_train_pca, y_train_ad)

# Best model extract
final_lgbm_model = rs.best_estimator_

print(f"Best Parameters: {rs.best_params_}")

run_model_evaluation("LightGBM", final_lgbm_model)

Best Parameters: {'num_leaves': 40, 'n_estimators': 100, 'learning_rate': 0.1, 'boosting_type': 'gbdt'}
--- For: LightGBM ---


,Fold,Classifier,Accuracy,Precision,Recall,Specificity,G-Mean,F1-Score,MCC,Kappa,ROC-AUC,Balanced Accuracy,Train_Time
0,1,LightGBM,0.9969,0.9969,0.9969,0.9990,0.9980,0.9969,0.9959,0.9959,0.9998,0.9969,25.4934
1,2,LightGBM,0.9964,0.9964,0.9964,0.9988,0.9976,0.9964,0.9952,0.9952,0.9998,0.9964,23.6473
2,3,LightGBM,0.9967,0.9967,0.9967,0.9989,0.9978,0.9967,0.9956,0.9956,0.9998,0.9967,23.4553
3,4,LightGBM,0.9964,0.9964,0.9964,0.9988,0.9976,0.9964,0.9952,0.9952,0.9999,0.9964,23.0135
4,5,LightGBM,0.9968,0.9968,0.9968,0.9989,0.9979,0.9968,0.9957,0.9957,0.9998,0.9968,22.7152
5,6,LightGBM,0.9964,0.9964,0.9964,0.9988,0.9976,0.9964,0.9952,0.9952,0.9998,0.9964,22.4513
6,7,LightGBM,0.9963,0.9963,0.9963,0.9988,0.9975,0.9963,0.9951,0.9951,0.9998,0.9963,21.1359
7,8,LightGBM,0.9965,0.9965,0.9965,0.9988,0.9977,0.9965,0.9953,0.9953,0.9998,0.9965,22.3949
8,9,LightGBM,0.9959,0.9959,0.9959,0.9986,0.9972,0.9959,0.9945,0.9945,0.9998,0.9959,22.4626
9,10,LightGBM,0.9960,0.9960,0.9960,0.9987,0.9973,0.9960,0.9947,0.9947,0.9998,0.9960,23.1339


# **Catboost**

In [11]:
from catboost import CatBoostClassifier
#  Model base initialize
model = CatBoostClassifier(random_state=42,verbose=0, thread_count=-1, loss_function='MultiClass')

# Parameter distribution setup
params = {
"iterations": [100, 200],
"learning_rate": [0.05, 0.1],
"depth": [4, 6]
}

#  Randomized Search execution
rs = RandomizedSearchCV(
    estimator=model,
    param_distributions=params,
    n_iter=2,
    cv=3,
    scoring='f1_weighted',
    n_jobs=-1,
    random_state=42
)

# Training
rs.fit(X_train_pca, y_train_ad)

# Best model extract
final_cat_model = rs.best_estimator_

print(f"Best Parameters: {rs.best_params_}")

run_model_evaluation("Catboost", final_cat_model)

Best Parameters: {'learning_rate': 0.1, 'iterations': 100, 'depth': 6}
--- For: Catboost ---


,Fold,Classifier,Accuracy,Precision,Recall,Specificity,G-Mean,F1-Score,MCC,Kappa,ROC-AUC,Balanced Accuracy,Train_Time
0,1,Catboost,0.9518,0.9525,0.9518,0.9839,0.9677,0.9518,0.9360,0.9358,0.9958,0.9518,28.0589
1,2,Catboost,0.9490,0.9497,0.9490,0.9830,0.9658,0.9489,0.9323,0.9320,0.9957,0.9490,27.9404
2,3,Catboost,0.9505,0.9512,0.9505,0.9835,0.9669,0.9504,0.9343,0.9340,0.9958,0.9505,27.7607
3,4,Catboost,0.9499,0.9507,0.9499,0.9833,0.9665,0.9499,0.9335,0.9332,0.9957,0.9499,27.7862
4,5,Catboost,0.9518,0.9525,0.9518,0.9839,0.9677,0.9518,0.9360,0.9358,0.9961,0.9518,28.7056
5,6,Catboost,0.9492,0.9500,0.9492,0.9831,0.9660,0.9491,0.9325,0.9322,0.9956,0.9492,28.0120
6,7,Catboost,0.9495,0.9502,0.9495,0.9832,0.9662,0.9495,0.9329,0.9327,0.9958,0.9495,27.7518
7,8,Catboost,0.9494,0.9500,0.9494,0.9831,0.9661,0.9493,0.9327,0.9325,0.9957,0.9494,27.7344
8,9,Catboost,0.9504,0.9510,0.9504,0.9835,0.9668,0.9503,0.9341,0.9338,0.9956,0.9504,27.7755
9,10,Catboost,0.9487,0.9494,0.9487,0.9829,0.9656,0.9486,0.9318,0.9316,0.9955,0.9487,28.4042


# **SVM**

In [12]:
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
#  Model base initialize
base_model = LinearSVC(random_state=42, max_iter=2000, dual=False)
model = CalibratedClassifierCV(base_model)

# Parameter distribution setup
params = {
"estimator__C": [0.1, 1, 10]
}

#  Randomized Search execution
rs = RandomizedSearchCV(
    estimator=model,
    param_distributions=params,
    n_iter=2,
    cv=3,
    scoring='f1_weighted',
    n_jobs=-1,
    random_state=42
)

# Training
rs.fit(X_train_pca, y_train_ad)

# Best model extract
final_svc_model = rs.best_estimator_

print(f"Best Parameters: {rs.best_params_}")

run_model_evaluation("SVM", final_svc_model)

Best Parameters: {'estimator__C': 1}
--- For: SVM ---


,Fold,Classifier,Accuracy,Precision,Recall,Specificity,G-Mean,F1-Score,MCC,Kappa,ROC-AUC,Balanced Accuracy,Train_Time
0,1,SVM,0.6719,0.6728,0.6719,0.8906,0.7736,0.6699,0.5640,0.5625,0.8873,0.6719,21.8163
1,2,SVM,0.6739,0.6747,0.6739,0.8913,0.7750,0.6720,0.5666,0.5652,0.8877,0.6739,27.0507
2,3,SVM,0.6701,0.6711,0.6701,0.8900,0.7723,0.6683,0.5615,0.5601,0.8863,0.6701,24.2696
3,4,SVM,0.6742,0.6744,0.6742,0.8914,0.7752,0.6724,0.5667,0.5656,0.8876,0.6742,20.9341
4,5,SVM,0.6711,0.6717,0.6711,0.8904,0.7730,0.6689,0.5629,0.5614,0.8879,0.6711,21.0982
5,6,SVM,0.6712,0.6720,0.6712,0.8904,0.7731,0.6692,0.5630,0.5616,0.8865,0.6712,21.4973
6,7,SVM,0.6683,0.6692,0.6683,0.8894,0.7710,0.6663,0.5593,0.5578,0.8853,0.6683,20.1693
7,8,SVM,0.6725,0.6734,0.6725,0.8908,0.7740,0.6706,0.5647,0.5633,0.8861,0.6725,20.2542
8,9,SVM,0.6719,0.6727,0.6719,0.8906,0.7736,0.6698,0.5640,0.5625,0.8860,0.6719,21.2231
9,10,SVM,0.6684,0.6691,0.6684,0.8895,0.7710,0.6664,0.5592,0.5578,0.8848,0.6684,20.1160


# **KNN**

In [13]:
from sklearn.neighbors import KNeighborsClassifier
#  Model base initialize
model = KNeighborsClassifier()

# Parameter distribution setup
params = {
"n_neighbors": [3, 5, 7],
"weights": ['uniform', 'distance'],
}

#  Randomized Search execution
rs = RandomizedSearchCV(
    estimator=model,
    param_distributions=params,
    n_iter=2,
    cv=3,
    scoring='f1_weighted',
    n_jobs=-1,
    random_state=42
)

# Training
rs.fit(X_train_pca, y_train_ad)

# Best model extract
final_knn_model = rs.best_estimator_

print(f"Best Parameters: {rs.best_params_}")

run_model_evaluation("KNN", final_knn_model)

Best Parameters: {'weights': 'distance', 'n_neighbors': 3}
--- For: KNN ---


,Fold,Classifier,Accuracy,Precision,Recall,Specificity,G-Mean,F1-Score,MCC,Kappa,ROC-AUC,Balanced Accuracy,Train_Time
0,1,KNN,0.9986,0.9986,0.9986,0.9995,0.9991,0.9986,0.9981,0.9981,0.9994,0.9986,1.5797
1,2,KNN,0.9985,0.9985,0.9985,0.9995,0.9990,0.9985,0.9979,0.9979,0.9994,0.9985,2.4506
2,3,KNN,0.9988,0.9988,0.9988,0.9996,0.9992,0.9988,0.9985,0.9985,0.9995,0.9988,1.4675
3,4,KNN,0.9987,0.9987,0.9987,0.9996,0.9991,0.9987,0.9982,0.9982,0.9994,0.9987,1.5222
4,5,KNN,0.9986,0.9986,0.9986,0.9995,0.9991,0.9986,0.9981,0.9981,0.9994,0.9986,1.5884
5,6,KNN,0.9982,0.9982,0.9982,0.9994,0.9988,0.9982,0.9976,0.9976,0.9992,0.9982,1.4607
6,7,KNN,0.9984,0.9984,0.9984,0.9995,0.9989,0.9984,0.9978,0.9978,0.9993,0.9984,1.4826
7,8,KNN,0.9987,0.9987,0.9987,0.9996,0.9992,0.9987,0.9983,0.9983,0.9996,0.9987,1.5032
8,9,KNN,0.9985,0.9985,0.9985,0.9995,0.9990,0.9985,0.9980,0.9980,0.9993,0.9985,2.5199
9,10,KNN,0.9982,0.9982,0.9982,0.9994,0.9988,0.9982,0.9976,0.9976,0.9993,0.9982,1.5178


# **MLP**

In [14]:
from sklearn.neural_network import MLPClassifier
#  Model base initialize
model = MLPClassifier(max_iter=200, random_state=42, early_stopping=True)

# Parameter distribution setup
params = {
"hidden_layer_sizes": [(64,),(64,32)],
            "alpha": [0.0001, 0.01]
}

#  Randomized Search execution
rs = RandomizedSearchCV(
    estimator=model,
    param_distributions=params,
    n_iter=2,
    cv=3,
    scoring='f1_weighted',
    n_jobs=-1,
    random_state=42
)

# Training
rs.fit(X_train_pca, y_train_ad)

# Best model extract
final_mlp_model = rs.best_estimator_

print(f"Best Parameters: {rs.best_params_}")

run_model_evaluation("MLP", final_mlp_model)

Best Parameters: {'hidden_layer_sizes': (64, 32), 'alpha': 0.0001}
--- For: MLP ---


,Fold,Classifier,Accuracy,Precision,Recall,Specificity,G-Mean,F1-Score,MCC,Kappa,ROC-AUC,Balanced Accuracy,Train_Time
0,1,MLP,0.9973,0.9973,0.9973,0.9991,0.9982,0.9972,0.9963,0.9963,0.9997,0.9973,53.0352
1,2,MLP,0.9971,0.9971,0.9971,0.9990,0.9981,0.9971,0.9962,0.9962,0.9996,0.9971,46.7360
2,3,MLP,0.9985,0.9985,0.9985,0.9995,0.9990,0.9985,0.9980,0.9980,0.9998,0.9985,122.5939
3,4,MLP,0.9977,0.9977,0.9977,0.9992,0.9985,0.9977,0.9970,0.9970,0.9998,0.9977,78.3814
4,5,MLP,0.9983,0.9983,0.9983,0.9994,0.9989,0.9983,0.9977,0.9977,0.9998,0.9983,105.2492
5,6,MLP,0.9972,0.9972,0.9972,0.9991,0.9981,0.9971,0.9962,0.9962,0.9997,0.9972,59.9378
6,7,MLP,0.9984,0.9984,0.9984,0.9995,0.9989,0.9983,0.9978,0.9978,0.9997,0.9984,126.0150
7,8,MLP,0.9982,0.9982,0.9982,0.9994,0.9988,0.9982,0.9976,0.9976,0.9998,0.9982,77.7274
8,9,MLP,0.9979,0.9979,0.9979,0.9993,0.9986,0.9979,0.9972,0.9972,0.9997,0.9979,69.3799
9,10,MLP,0.9985,0.9985,0.9985,0.9995,0.9990,0.9985,0.9980,0.9980,0.9998,0.9985,102.5733


# **Bagging**

In [32]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import RandomizedSearchCV


X_train_pca_fast = X_train_pca.astype('float32')


base_tree = DecisionTreeClassifier(max_depth=5, random_state=42)

#  Model base initialize
model = BaggingClassifier(
    estimator=base_tree,
    random_state=42
)

# Parameter distribution setup
params = {
    "n_estimators": [10, 50],
    "max_samples": [0.5, 1.0],
}

# Randomized Search execution
rs = RandomizedSearchCV(
    estimator=model,
    param_distributions=params,
    n_iter=2,
    cv=3,
    scoring='f1_weighted',
    n_jobs=-1,
    random_state=42
)



rs.fit(X_train_pca_fast, y_train_ad)

# Best model extract
final_bg_model = rs.best_estimator_

print(f"Best Parameters: {rs.best_params_}")

# Model evaluation call
run_model_evaluation("Bagging", final_bg_model)

Best Parameters: {'n_estimators': 50, 'max_samples': 1.0}
--- For: Bagging ---


,Fold,Classifier,Accuracy,Precision,Recall,Specificity,G-Mean,F1-Score,MCC,Kappa,ROC-AUC,Balanced Accuracy,Train_Time
0,1,Bagging,0.7240,0.7296,0.7240,0.9080,0.8108,0.7211,0.6348,0.6320,0.8947,0.7240,299.0090
1,2,Bagging,0.7179,0.7244,0.7179,0.9060,0.8065,0.7139,0.6274,0.6239,0.8989,0.7179,304.8226
2,3,Bagging,0.7120,0.7155,0.7120,0.9040,0.8022,0.7083,0.6187,0.6160,0.8996,0.7120,365.5737
3,4,Bagging,0.7125,0.7165,0.7125,0.9042,0.8026,0.7086,0.6196,0.6166,0.8918,0.7125,315.0667
4,5,Bagging,0.7119,0.7158,0.7119,0.9040,0.8022,0.7079,0.6189,0.6159,0.8955,0.7119,293.8873
5,6,Bagging,0.7222,0.7287,0.7222,0.9074,0.8095,0.7189,0.6329,0.6296,0.9006,0.7222,292.0061
6,7,Bagging,0.7215,0.7276,0.7214,0.9072,0.8090,0.7188,0.6314,0.6286,0.8939,0.7214,292.4466
7,8,Bagging,0.7130,0.7176,0.7130,0.9043,0.8030,0.7092,0.6203,0.6173,0.8973,0.7130,291.6137
8,9,Bagging,0.7118,0.7163,0.7118,0.9039,0.8021,0.7079,0.6187,0.6157,0.8917,0.7118,291.2967
9,10,Bagging,0.7133,0.7175,0.7133,0.9044,0.8032,0.7102,0.6203,0.6177,0.8950,0.7133,292.3544


# **Stacking**

In [35]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
base_learners = [
    ('xgb', final_xg_model),
    ('lgbm', final_lgbm_model)
]

meta_model = LogisticRegression(max_iter=1000)

stack_model = StackingClassifier(
    estimators=base_learners,
    final_estimator=meta_model,
    cv=2,
    n_jobs=-1,
    passthrough=False
)
stack_model.fit(X_train_pca, y_train_ad)
final_stack_model = stack_model

run_model_evaluation("Stacking Classifier", final_stack_model)

--- For: Stacking Classifier ---


,Fold,Classifier,Accuracy,Precision,Recall,Specificity,G-Mean,F1-Score,MCC,Kappa,ROC-AUC,Balanced Accuracy,Train_Time
0,1,Stacking Classifier,0.9920,0.9921,0.9920,0.9973,0.9947,0.9920,0.9894,0.9894,0.9997,0.9920,103.5055
1,2,Stacking Classifier,0.9913,0.9914,0.9913,0.9971,0.9942,0.9912,0.9884,0.9884,0.9994,0.9913,100.7987
2,3,Stacking Classifier,0.9919,0.9920,0.9919,0.9973,0.9946,0.9919,0.9893,0.9893,0.9997,0.9919,106.7001
3,4,Stacking Classifier,0.9914,0.9915,0.9914,0.9971,0.9943,0.9913,0.9886,0.9885,0.9996,0.9914,100.9601
4,5,Stacking Classifier,0.9918,0.9919,0.9918,0.9973,0.9946,0.9918,0.9892,0.9891,0.9996,0.9918,100.8111
5,6,Stacking Classifier,0.9909,0.9910,0.9909,0.9970,0.9939,0.9909,0.9880,0.9879,0.9997,0.9909,101.2914
6,7,Stacking Classifier,0.9909,0.9910,0.9909,0.9970,0.9939,0.9909,0.9880,0.9879,0.9995,0.9909,101.6516
7,8,Stacking Classifier,0.9915,0.9916,0.9915,0.9972,0.9943,0.9915,0.9887,0.9887,0.9996,0.9915,101.5340
8,9,Stacking Classifier,0.9906,0.9907,0.9906,0.9969,0.9937,0.9905,0.9875,0.9874,0.9996,0.9906,100.9807
9,10,Stacking Classifier,0.9905,0.9906,0.9905,0.9968,0.9937,0.9904,0.9874,0.9873,0.9995,0.9905,102.0677


In [36]:
!pip install xlsxwriter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 6.8 MB/s eta 0:00:00


In [44]:
import pandas as pd

df = pd.DataFrame(results)

df['Classifier'] = df['Classifier'].replace({
    'catboost': 'CatBoost',
    'Catboost': 'CatBoost',
    'XGBoost': 'xgboost',
    'xgb': 'xgboost',
    'lightgbm': 'LightGBM',
    'knn': 'KNN',
    'Stacking': 'Stacking Classifier'
})

col_order = [
    'Logistic Regression', 'Decision Tree', 'Random Forest', 'GBM',
    'xgboost', 'LightGBM', 'CatBoost', 'SVM', 'KNN', 'MLP', 'Bagging', 'Stacking Classifier'
]

metrics = ["Accuracy", "Precision", "Recall", "Specificity", "F1 Score", "ROC-AUC", "MCC", "G-Mean", "Kappa"]

with pd.ExcelWriter('ADASYN_KFold_Formate_Result(INCART 2-lead Arrhythmia Database).xlsx') as writer:
    for metric in metrics:

        actual_col = metric if metric in df.columns else "F1-Score" if metric == "F1 Score" else "G-Mean" if metric == "G-Mean" else None

        if actual_col and actual_col in df.columns:

            pivot_df = df.pivot(index='Fold', columns='Classifier', values=actual_col)

            valid_cols = [c for c in col_order if c in pivot_df.columns]
            pivot_df = pivot_df[valid_cols]

            pivot_df.to_excel(writer, sheet_name=metric)